# 02 · Fase A — Teardown de DETR-R50
Parámetros y FLOPs por bloque, latencia/memoria, características intermedias, mapas de atención y campo receptivo.
*FLOPs = 2 × MACs (convención de `torch.utils.flop_counter`).*

In [ ]:
# Colab/Kaggle: clona el repo y instala dependencias (ajusta la URL)
# !git clone https://github.com/<usuario>/detr-query-saturation.git && cd detr-query-saturation
# !pip -q install -r requirements.txt
import sys; sys.path.append('../src' if __import__('os').path.exists('../src') else 'src')
from common import *
import torch, json, os, pandas as pd, matplotlib.pyplot as plt
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; SEED=42; set_seed(SEED); print(DEVICE)

In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor
from PIL import Image
CKPT='facebook/detr-resnet-50'
processor=DetrImageProcessor.from_pretrained(CKPT)
model=DetrForObjectDetection.from_pretrained(CKPT).to(DEVICE).eval()
IMG_PATH='../data/ejemplo_visdrone.jpg'   # una imagen densa de VisDrone
img=Image.open(IMG_PATH).convert('RGB'); enc=processor(images=img,return_tensors='pt')
pv,pm=enc['pixel_values'].to(DEVICE),enc['pixel_mask'].to(DEVICE); print(pv.shape)

## A1 · Parámetros por bloque

In [ ]:
blocks={'backbone (ResNet-50)':model.model.backbone,'input_projection':model.model.input_projection,
 'query_embeddings':model.model.query_position_embeddings,'encoder (6 capas)':model.model.encoder,
 'decoder (6 capas)':model.model.decoder,'class head':model.class_labels_classifier,'box head (MLP)':model.bbox_predictor}
P=pd.DataFrame([(k,sum(p.numel() for p in m.parameters())) for k,m in blocks.items()],columns=['bloque','parámetros'])
P['%']=(100*P['parámetros']/P['parámetros'].sum()).round(1); print(P); print('total',P['parámetros'].sum())
P.to_csv('../results/params_por_bloque.csv',index=False)

## A2 · FLOPs por bloque

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
with torch.no_grad(), FlopCounterMode(model,depth=4,display=False) as fc: model(pixel_values=pv,pixel_mask=pm)
counts={k:sum(v.values()) for k,v in fc.get_flop_counts().items()}
for k,v in counts.items(): print(f'{v/1e9:9.2f} GFLOPs  {k}')   # revisa los nombres y ajusta los filtros si hace falta
rows=[]
for key in ['model.backbone','model.input_projection','model.encoder','model.decoder','class_labels_classifier','bbox_predictor']:
    rows.append((key,sum(v for k,v in counts.items() if k.endswith(key))/1e9))
F=pd.DataFrame(rows,columns=['bloque','GFLOPs']); F['%']=(100*F['GFLOPs']/fc.get_total_flops()*1e9).round(1)
print(F,'\ntotal',fc.get_total_flops()/1e9,'GFLOPs'); F.to_csv('../results/flops_por_bloque.csv',index=False)

## A3 · Latencia y memoria (calentamiento + repeticiones)

In [ ]:
import time
def bench(fn,runs=30,warm=10):
    for _ in range(warm): fn()
    torch.cuda.synchronize(); t=time.time()
    for _ in range(runs): fn()
    torch.cuda.synchronize(); return (time.time()-t)/runs*1000
assert DEVICE=='cuda','medir latencia en GPU'
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    lat=bench(lambda: model(pixel_values=pv,pixel_mask=pm))
    # por bloque (aprox.): hooks con sincronización
    times={}
    def mk(name):
        def pre(m,i): torch.cuda.synchronize(); times[name]=-time.time()
        def post(m,i,o): torch.cuda.synchronize(); times[name]+=time.time()
        return pre,post
    for name,m in {'backbone':model.model.backbone,'encoder':model.model.encoder,'decoder':model.model.decoder}.items():
        a,b=mk(name); m.register_forward_pre_hook(a); m.register_forward_hook(b)
    model(pixel_values=pv,pixel_mask=pm)
print(f'latencia total {lat:.1f} ms | memoria pico {torch.cuda.max_memory_allocated()/2**20:.0f} MiB')
print({k:round(v*1000,1) for k,v in times.items()},'ms (una pasada, aprox.)')

## A4 · Características intermedias del backbone

In [ ]:
with torch.no_grad(): feats=model.model.backbone.conv_encoder(pv,pm)
fig,ax=plt.subplots(1,len(feats)+1,figsize=(4*(len(feats)+1),4)); ax[0].imshow(img); ax[0].set_title('imagen')
for i,(f,_) in enumerate(feats): ax[i+1].imshow(f[0].mean(0).cpu(),cmap='viridis'); ax[i+1].set_title(f'etapa {i+1}: {tuple(f.shape[1:])}')
for a in ax: a.axis('off')
plt.savefig('../results/features_intermedias.png',dpi=150,bbox_inches='tight')

## A5 · Mapas de atención del decoder (últimas capas, queries con mayor score)

In [ ]:
with torch.no_grad(): out=model(pixel_values=pv,pixel_mask=pm,output_attentions=True)
(features,_)=model.model.backbone(pv,pm); h,w=features[-1][0].shape[-2:]
att=out.cross_attentions[-1][0].mean(0)             # (queries, h*w), promedio de cabezas
top=out.logits[0].softmax(-1)[:,:-1].max(-1).values.topk(5).indices.tolist()
fig,ax=plt.subplots(1,5,figsize=(20,4))
for a,q in zip(ax,top): a.imshow(att[q].reshape(h,w).cpu()); a.set_title(f'query {q}'); a.axis('off')
plt.savefig('../results/atencion_decoder.png',dpi=150,bbox_inches='tight')

## A6 · Campo receptivo (efectivo, por gradiente) vs teórico (≈483 px en ResNet-50, capa final)

In [ ]:
x=pv.clone().requires_grad_(True)
fm=model.model.backbone(x,pm)[0][-1][0]; cy,cx=fm.shape[-2]//2,fm.shape[-1]//2
fm[0,:,cy,cx].sum().backward()
erf=x.grad.abs().sum(1)[0].cpu(); plt.imshow(erf,cmap='magma'); plt.title('campo receptivo efectivo (centro del mapa stride-32)')
plt.savefig('../results/campo_receptivo.png',dpi=150,bbox_inches='tight')
ys,xs=torch.where(erf>erf.max()*0.05); print('extensión efectiva ≈',(ys.max()-ys.min()).item(),'x',(xs.max()-xs.min()).item(),'px')

## A7 · Reproducción de la línea base en COCO val2017 (subconjunto)
Descarga val2017 + anotaciones (Colab) y evalúa ~500 imágenes. Esperado: AP cercano a 42 en el val completo (reportado por Carion et al.). Documenta la diferencia.

In [ ]:
# !wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip -d ../data
# !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip -d ../data
make_subset('../data/annotations/instances_val2017.json',500,'../data/coco_val_sub.json',SEED)
dets=predict(model,processor,'../data/val2017','../data/coco_val_sub.json',device=DEVICE,label_offset=0,shift=0)
print(coco_eval('../data/coco_val_sub.json',dets))